In [1]:

from pprint import pprint

from ollama import chat
from tools import Top_Type, code_extraction, equi_check, linter, netlist
# from typing import TypedDict
# from langgraph.graph import START,StateGraph,END


In [2]:
Ollama_MODEL="qwen2.5-coder:7b"

# Checking for the presence of model and loading  the model 


In [3]:
stream=chat(model=Ollama_MODEL,
            messages=[{"role":"user","content":"What is your knowledge cut off and how  good are you with verilog and symbolic model verification coding ?"}],
            stream=True)
print("Printing the answer\n")
for chunk in stream:
    print(chunk['message']['content'],end='',flush=True)
print()

Printing the answer

My knowledge cutoff date is October 2023. I can provide information and assistance based on data up to that point.

Regarding Verilog and symbolic model verification, I can offer general guidance and help with understanding basic concepts and techniques. However, for specific coding problems or complex issues, it might be more effective to consult a specialized resource or seek advice from someone with extensive experience in these areas. If you have a particular question or need help with a specific piece of code, feel free to ask, and I'll do my best to assist you based on the information available up to my cutoff date.


In [ ]:
# SPECIFICATION_SYSTEM_PROMPT = r"""
# R — ROLE
# --------
# You are a Hardware Design Specification Provider.

# Convert the user's natural-language RTL request into a structured,
# detailed specification for a separate Verilog-2001 code-generation LLM.

# You do NOT generate RTL.

# I — INSTRUCTIONS
# -----------------
# Extract the intended:

# - design/function
# - combinational or sequential behavior
# - inputs/outputs, widths and signedness
# - clock/reset requirements
# - protocol/interface behavior
# - operations and data flow
# - state/FSM behavior
# - timing/latency requirements
# - important corner cases

# Do not invent critical functionality. Clearly identify assumptions
# and missing information.

# S — STEPS
# ---------
# 1. Identify the design and its purpose.
# 2. Extract the complete interface.
# 3. Determine combinational/sequential behavior.
# 4. Define functional and control behavior.
# 5. Define clock/reset behavior.
# 6. Define states/protocol behavior where applicable.
# 7. Identify assumptions, ambiguities and corner cases.
# 8. Provide implementation guidance for the RTL generator.

# E — EXPECTED OUTPUT
# -------------------
# Return ONLY valid JSON:

# {
#   "design": {
#     "name": "",
#     "type": "COMBINATIONAL | SEQUENTIAL",
#     "purpose": "",
#     "description": ""
#   },

#   "interface": [
#     {
#       "name": "",
#       "direction": "input | output | inout",
#       "width": "",
#       "signed": "yes | no | unspecified",
#       "description": ""
#     }
#   ],

#   "clock_reset": {
#     "clock": "",
#     "frequency": "",
#     "period": "",
#     "reset": "",
#     "reset_type": "synchronous | asynchronous | unspecified",
#     "polarity": "active_high | active_low | unspecified"
#   },

#   "functional_specification": {
#     "main_function": "",
#     "operations": [],
#     "data_flow": [],
#     "control_flow": [],
#     "state_behavior": [],
#     "protocol_behavior": [],
#     "latency": ""
#   },

#   "corner_cases": [],
#   "assumptions": [],
#   "clarifications": [],

#   "code_generation_guidance": {
#     "language": "Verilog-2001",
#     "module_name": "",
#     "requirements": []
#   }
# }

# N — NARROW CONSTRAINTS
# ----------------------
# 1. Output ONLY JSON.
# 2. Do not generate RTL or testbench code.
# 3. Do not discuss tools or verification flows.
# 4. Do not invent unspecified critical behavior.
# 5. Separate explicit requirements from assumptions.
# 6. Preserve all important information from the user.
# 7. Make the functional specification detailed enough for the RTL
#    generator to implement it without reinterpreting the original prompt.
# 8. Use Verilog-2001; do not specify SystemVerilog constructs.
# 9. Do not output hidden reasoning or chain-of-thought.
# 10. Keep the output focused on the design specification.
# """

In [22]:
SPECIFICATION_SYSTEM_PROMPT = r"""
ROLE  
You are an RTL specification extractor.

INSTRUCTIONS  
Convert the user's request into the minimum information required to generate synthesizable Verilog-2001 RTL.  
Extract only explicitly stated requirements. Never invent behavior.

STEPS  
1. Identify module name and COMBINATIONAL/SEQUENTIAL type.  
2. Extract exact ports, directions, widths and signedness when stated.  
3. Extract clock/reset only when stated.  
4. Convert required functionality into short, unambiguous behavior rules.  
5. List FSM states only when explicitly required or clearly defined.  
6. Preserve numeric parameters exactly.

END GOAL  
Return one compact machine-readable RTL contract for the code generator.

NARROWING CONSTRAINTS  
- JSON only; no Markdown or explanation.
- Do not add assumptions, corner cases, optimizations, timing, latency, protocols or error handling unless explicitly specified.
- Do not duplicate information.
- Do not infer missing widths, resets, states or functionality.
- Unknown values = null.
- Use exact user signal/module names.
- Each behavior item must describe one implementation requirement.
- No RTL/code generation.

OUTPUT
{
  "module": "",
  "type": "COMBINATIONAL|SEQUENTIAL",
  "ports": [
    {"name":"","dir":"input|output|inout","width":null,"signed":null}
  ],
  "clock": {"name":null,"edge":null},
  "reset": {"name":null,"edge":null,"async":null},
  "parameters": {},
  "behavior": [],
  "states": []
}
"""

In [23]:
def spec_analyzer(prompt:str):
    stream=chat(model=Ollama_MODEL,messages=[{"role":"System",
                                              "content":f""" {SPECIFICATION_SYSTEM_PROMPT} """},
                                              {"role":"user",
                                               "content":f"{prompt}"}],
                                               stream=True,
                                               options={
                                                        "temperature":0.25,
                                                        "top_p":0.9,
                                                        "top_k":20,
                                                        "seed":0
                                                      })
    respon=""
    for chunk in stream:
        content=chunk["message"]["content"] 
        print(content,end="",flush=True)
        respon+=content
    return respon

In [24]:
_VERILOG_SYSTEM =""" 
ROLE
You are a senior RTL design engineer specializing in synthesizable Verilog-2001.

INSTRUCTIONS
Implement the user's specification exactly. Generate clean, synthesizable RTL.
Use Verilog-2001 only and return only the complete Verilog code.

STEPS

1. Identify all signals, their widths, signedness, clocks, resets, and required behavior.
2. Generate the RTL using correct sequential and combinational coding styles.
3. Before returning, check syntax, widths, constants, signedness, bit selections, and synthesis safety.

END GOAL
Produce Verilator/Icarus-compatible Verilog-2001 RTL with no width or syntax errors.

NARROW CONSTRAINTS

* Use explicit signal widths.
* Every sized numeric literal must have enough bits for all specified digits.
  Example: 3'b111, NOT 2'b111.
* Match constant widths to the destination/comparison signal where practical.
  Example: reg [2:0] count → 3'b111 or 3'd7.
* Never select bits outside a signal's declared range.
* Use '=' in always @(*) and '<=' in clocked always blocks.
* Give combinational outputs a value on every path.
* Include default in case statements.
* Use $signed/$unsigned when signedness matters.
* No SystemVerilog: logic, always_comb, always_ff, typedef, enum, struct, interface, class, etc.
* Do not invent requirements or modify the user's specification.
* Return ONLY Verilog-2001 code; no Markdown or explanation.

"""

In [25]:
def code_gen(prompt:str)->str:
    """ 
    INPUT:
    prompt: excepts user promtp a string providing the detials about the verilog code to be written by the llm.

    OUTPTS:
        gives the code as output 
    """
    stream=chat(model=Ollama_MODEL,messages=[
                                                {
                                                    "role": "system",
                                                    "content": _VERILOG_SYSTEM
                                                },
                                                {
                                                    "role": "user",
                                                    "content": prompt
                                                }
                                            ],
                                            stream=True,
                                            options={
                                                    "temperature":0.45,
                                                     "top_p":0.8,
                                                     "top_k":20,
                                                     "seed":0
                                                     })
    resp=""
    for chunk in stream:
        content=chunk["message"]["content"]
        print(content,end="",flush=True)
        resp+=content
    return resp

In [26]:
User_inpt=input("Prompt the verilog module  requesting")
spec_resp=spec_analyzer(User_inpt)


```json
{
  "module": "uart_rx",
  "type": "COMBINATIONAL|SEQUENTIAL",
  "ports": [
    {"name":"clk","dir":"input","width":null,"signed":null},
    {"name":"rst_n","dir":"input","width":null,"signed":null},
    {"name":"rx","dir":"input","width":1,"signed":null},
    {"name":"data_out","dir":"output","width":32,"signed":false},
    {"name":"valid","dir":"output","width":1,"signed":null}
  ],
  "clock": {"name":"clk","edge":"posedge"},
  "reset": {"name":"rst_n","edge":"negedge","async":true},
  "parameters": {
    "BAUD_RATE": 9600
  },
  "behavior": [
    "Parameter BAUD_PERIOD = (100_000_000 / BAUD_RATE);",
    "Register bit_count : [2:0] := 3'b000;",
    "Register data_reg : [31:0] := 32'b0;",
    "Register sample_point : [7:0] := 8'hFF;",
    "Register clk_div : [7:0] := 8'b0;",
    "Register rx_prev : [0:0] := 1'b1;",
    "Register valid_reg : [0:0] := 1'b0;",
    "Sequential always @(posedge clk or negedge rst_n) begin",
      "if (!rst_n) begin",
        "bit_count <= 3'b000;",

In [27]:
response=code_gen(spec_resp)

```verilog
module uart_rx (
  input wire clk,
  input wire rst_n,
  input wire rx,
  output reg [31:0] data_out,
  output reg valid
);

parameter BAUD_RATE = 9600;
parameter BAUD_PERIOD = (100_000_000 / BAUD_RATE);
reg [2:0] bit_count;
reg [31:0] data_reg;
reg [7:0] sample_point;
reg [7:0] clk_div;
reg rx_prev;
reg valid_reg;

always @(posedge clk or negedge rst_n) begin
  if (!rst_n) begin
    bit_count <= 3'b000;
    data_reg <= 32'b0;
    sample_point <= 8'hFF;
    clk_div <= 8'b0;
    rx_prev <= 1'b1;
    valid_reg <= 1'b0;
  end else begin
    if (clk_div == BAUD_PERIOD - 1) begin
      clk_div <= 8'b0;
      sample_point <= sample_point + 1;
    end else begin
      clk_div <= clk_div + 1;
    end
    if (sample_point == 8'hFF) begin
      data_reg[bit_count] <= rx;
      bit_count <= bit_count + 1;
      sample_point <= 8'b0;
    end
    if (bit_count == 32) begin
      valid_reg <= 1'b1;
    end else begin
      valid_reg <= 1'b0;
    end
  end
end

assign data_out = data_reg;


In [28]:
clean_Vcode=code_extraction(response)
print(clean_Vcode)

module uart_rx (
  input wire clk,
  input wire rst_n,
  input wire rx,
  output reg [31:0] data_out,
  output reg valid
);

parameter BAUD_RATE = 9600;
parameter BAUD_PERIOD = (100_000_000 / BAUD_RATE);
reg [2:0] bit_count;
reg [31:0] data_reg;
reg [7:0] sample_point;
reg [7:0] clk_div;
reg rx_prev;
reg valid_reg;

always @(posedge clk or negedge rst_n) begin
  if (!rst_n) begin
    bit_count <= 3'b000;
    data_reg <= 32'b0;
    sample_point <= 8'hFF;
    clk_div <= 8'b0;
    rx_prev <= 1'b1;
    valid_reg <= 1'b0;
  end else begin
    if (clk_div == BAUD_PERIOD - 1) begin
      clk_div <= 8'b0;
      sample_point <= sample_point + 1;
    end else begin
      clk_div <= clk_div + 1;
    end
    if (sample_point == 8'hFF) begin
      data_reg[bit_count] <= rx;
      bit_count <= bit_count + 1;
      sample_point <= 8'b0;
    end
    if (bit_count == 32) begin
      valid_reg <= 1'b1;
    end else begin
      valid_reg <= 1'b0;
    end
  end
end

assign data_out = data_reg;
assign vali

In [29]:
res=linter(clean_Vcode)
print(res)

{'return_code': 0, 'stdout': '- V e r i l a t i o n   R e p o r t: Verilator 5.032 2025-01-01 rev (Debian 5.032-1)\n- Verilator: Built from 0.021 MB sources in 2 modules, into 0.015 MB in 5 C++ files needing 0.000 MB\n- Verilator: Walltime 0.005 s (elab=0.000, cvt=0.001, bld=0.000); cpu 0.004 s on 1 threads; alloced 7.883 MB\n', 'stderr': '%Warning-WIDTHEXPAND: /tmp/tmpq0j1_9ml.v:27:17: Operator EQ expects 32 or 27 bits on the LHS, but LHS\'s VARREF \'clk_div\' generates 8 bits.\n                                              : ... note: In instance \'uart_rx\'\n   27 |     if (clk_div == BAUD_PERIOD - 1) begin\n      |                 ^~\n                      ... For warning description see https://verilator.org/warn/WIDTHEXPAND?v=5.032\n                      ... Use "/* verilator lint_off WIDTHEXPAND */" and lint_on around source to disable this message.\n%Warning-WIDTHEXPAND: /tmp/tmpq0j1_9ml.v:34:15: Bit extraction of var[31:0] requires 5 bit index, not 3 bits.\n                   

In [30]:
top=Top_Type(clean_Vcode)

In [31]:
pprint(top)

{'err': '',
 'output': '\n'
           ' '
           '/----------------------------------------------------------------------------\\\n'
           ' |  yosys -- Yosys Open SYnthesis '
           'Suite                                       |\n'
           ' |  Copyright (C) 2012 - 2026  Claire Xenia Wolf '
           '<claire@yosyshq.com>         |\n'
           ' |  Distributed under an ISC-like license, type "license" to see '
           'terms        |\n'
           ' '
           '\\----------------------------------------------------------------------------/\n'
           ' Yosys 0.62+112 (git sha1 687a36af3, g++ 13.3.0-6ubuntu2~24.04.1 '
           '-fPIC -O3)\n'
           '\n'
           '-- Running command ` \n'
           '                             read_verilog /tmp/tmp_f8meyll.v;\n'
           '                             proc;\n'
           '                             stat;\n'
           "                            ' --\n"
           '\n'
           '1. Executing V

In [32]:
top_name=top['top_module']
print(top_name)

uart_rx


In [33]:
net_out=netlist(top=top_name,module_path=f"./src/{top_name}.v",lib_path="../Nangate45/Nangate45_typ.lib")

In [34]:
pprint(net_out)

{'netlist_returncode': 0,
 'netlist_stderr': '',
 'netlist_stdout': '\n'
                   ' '
                   '/----------------------------------------------------------------------------\\\n'
                   ' |  yosys -- Yosys Open SYnthesis '
                   'Suite                                       |\n'
                   ' |  Copyright (C) 2012 - 2026  Claire Xenia Wolf '
                   '<claire@yosyshq.com>         |\n'
                   ' |  Distributed under an ISC-like license, type "license" '
                   'to see terms        |\n'
                   ' '
                   '\\----------------------------------------------------------------------------/\n'
                   ' Yosys 0.62+112 (git sha1 687a36af3, g++ '
                   '13.3.0-6ubuntu2~24.04.1 -fPIC -O3)\n'
                   '\n'
                   '-- Running command `\n'
                   '            read_verilog ./src/uart_rx.v;\n'
                   '            hierarchy -top

In [35]:
path_1=f"./src/{top_name}.v"
path_2=f"./src/{top_name}_netlist.v"
des_type=top["type"]
equi_out=equi_check(top=top_name,path_1=path_1,path_2=path_2,design_type=des_type)

In [36]:
pprint(equi_out)

{'pass': 'Pass',
 'return_code': 0,
 'stderr': '',
 'stdout': '\n'
           ' '
           '/----------------------------------------------------------------------------\\\n'
           ' |  yosys -- Yosys Open SYnthesis '
           'Suite                                       |\n'
           ' |  Copyright (C) 2012 - 2026  Claire Xenia Wolf '
           '<claire@yosyshq.com>         |\n'
           ' |  Distributed under an ISC-like license, type "license" to see '
           'terms        |\n'
           ' '
           '\\----------------------------------------------------------------------------/\n'
           ' Yosys 0.62+112 (git sha1 687a36af3, g++ 13.3.0-6ubuntu2~24.04.1 '
           '-fPIC -O3)\n'
           '\n'
           '-- Running command `\n'
           '            read_verilog ./src/uart_rx.v;\n'
           '            prep -top uart_rx;\n'
           '            rename uart_rx gold;\n'
           '            design -stash rtl;\n'
           '            design -